# Modelo em ONNX para rodar no navegador

Converte o ConvNeXt-Tiny ajustado para ONNX, quantiza as camadas lineares em int8 e sobe os dois arquivos para a pasta `onnx/` do mesmo repositório no Hugging Face. É o formato que o Transformers.js lê, então a demo roda no navegador de quem visita, sem servidor.

As camadas lineares concentram quase todos os pesos do ConvNeXt. As convoluções ficam em fp32, porque o operador de convolução quantizada não roda no WebAssembly do navegador.

Roda em CPU, não precisa de GPU.

In [ ]:
!pip install -q "optimum-onnx[onnxruntime]==0.1.0" "datasets==5.1.0"

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
REPO = "juan1211/pratos-convnext-tiny-food101"
SAIDA = "onnx"
AMOSTRA = 500

!optimum-cli export onnx --model {REPO} --task image-classification {SAIDA}

## Quantização

In [ ]:
from pathlib import Path

from onnxruntime.quantization import QuantType, quantize_dynamic

quantize_dynamic(
    f"{SAIDA}/model.onnx",
    f"{SAIDA}/model_quantized.onnx",
    op_types_to_quantize=["MatMul", "Gemm"],
    weight_type=QuantType.QUInt8,
)
for nome in ("model.onnx", "model_quantized.onnx"):
    print(nome, round(Path(SAIDA, nome).stat().st_size / 2**20, 1), "MB")

## Conferência

O modelo em ONNX tem que dar o mesmo primeiro palpite que o PyTorch. A conferência usa fotos do teste do Food-101.

In [ ]:
import numpy as np
import onnxruntime as ort
import torch
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification

processador = AutoImageProcessor.from_pretrained(REPO)
original = AutoModelForImageClassification.from_pretrained(REPO).eval()
fotos = load_dataset("ethz/food101", split=f"validation[:{AMOSTRA}]")
entradas = processador([f.convert("RGB") for f in fotos["image"]], return_tensors="np")["pixel_values"]

with torch.inference_mode():
    palpite_original = original(pixel_values=torch.from_numpy(entradas)).logits.argmax(1).numpy()

for nome in ("model.onnx", "model_quantized.onnx"):
    sessao = ort.InferenceSession(f"{SAIDA}/{nome}")
    palpite = np.concatenate(
        [sessao.run(None, {"pixel_values": entradas[i : i + 50]})[0].argmax(1) for i in range(0, AMOSTRA, 50)]
    )
    acerto = (palpite == np.array(fotos["label"])).mean()
    print(f"{nome}: mesmo palpite do PyTorch em {(palpite == palpite_original).mean():.1%}, top-1 {acerto:.1%}")

## Subir para o Hugging Face

In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    repo_id=REPO,
    folder_path=SAIDA,
    path_in_repo="onnx",
    allow_patterns=["model.onnx", "model_quantized.onnx"],
    commit_message="pesos em ONNX para o Transformers.js",
)